# Minimal demo

In [ ]:
%load_ext autoreload
%autoreload 2

import mne
from pathlib import Path
from mne.preprocessing import find_eog_events

edf_path_demo = Path("data/real_data/test_from_20s.edf")
raw_demo = mne.io.read_raw_edf(edf_path_demo, preload=True, verbose=False)
raw_demo.drop_channels(['ECG  ECG'])

events_demo = find_eog_events(raw_demo, ch_name='EEG FP1-A1')

In [ ]:
import matplotlib
matplotlib.use('TkAgg')

raw_demo.plot(
    events=events_demo,
    duration=30,  # Show 30 seconds initially
    start=0,
    n_channels=15,
    scalings='auto',
    show=True,
)

# Test processing

In [ ]:
import numpy as np

def save_txt(raw: mne.io.BaseRaw, events: np.ndarray, out_path: Path) -> None:
    times: np.ndarray = (events[:, 0] - raw.first_samp) / raw.info['sfreq']
    np.savetxt(str(out_path), times, fmt="%.4f")

save_txt(raw_demo, events_demo, edf_path_demo.parent / "fp1_eyem.txt")

In [ ]:
def process(edf_path: Path, out_dir: Path | None = None, delta_sec: float = 0.05) -> None:
    """
    1. finds all eog events in FP1 and FP2 channels, saves to TXT
    2. merges two events list using `delta_sec` as max diff
    """
    if out_dir is None:
        out_dir = edf_path.parent
    
    raw = mne.io.read_raw_edf(edf_path, preload=True, verbose=False)
    chans = ["FP1-A1", "FP2-A2"]
    chans_to_events = {}
    for ch in chans:
        events = find_eog_events(raw, ch_name=f"EEG {ch}")
        chans_to_events[ch] = events
        save_txt(raw, events, out_dir / f"{edf_path.stem}_eog_{ch}.txt")
    
    # TODO merge two and save to `..._eog_consensus.txt`


process(edf_path_demo)


# Real processing

In [ ]:
big_edf_paths = [
    "/mnt/y/vzuev/datasets/eeg_videos/vkuklin_july19_2025/vkuklin_july19_2025 18-JUL-2025_21h28m18.382s.edf"
]